In [ ]:
import sys # for automation and parallelization: set manual to false when run by a launcher
import json
MANUAL, SCENARIO = (True, 'base') if 'ipykernel' in sys.argv[0]\
     else (False,  json.loads(sys.argv[1])['scenario'])

# START

In [2]:
sys.path.insert(0, r'../../../quetzal')

from quetzal.model import connectionscanmodel, stepmodel
from quetzal.io import excel
import numpy as np
import pandas as pd
import geopandas as gpd

import shapely

import matplotlib.pyplot as plt
from tqdm import tqdm

In [3]:
qpath = r'../../'
ipath = qpath + r'inputs/'
mpath = qpath + r'scenarios/{}/models/'.format(SCENARIO)

In [4]:
var = excel.read_var(ipath + r'parameters.xlsx', scenario=SCENARIO)

Index(['description', 'unit', 'category', 'parameter', 'base',
       'base_timetables'],
      dtype='object')


In [5]:
PARKING_FARES = var['fare_parking'].to_dict()

# READ

In [6]:
sm = stepmodel.read_zippedpickles(mpath + 'all_networks')

walk_on_road: 100%|██████████| 11/11 [00:00<00:00, 26.90it/s]


In [7]:
# TODO Louison: remove zone inputs from gitignore! -> résolu
zones = gpd.read_file(ipath + 'zones/final/zonage.gpkg').to_crs(epsg='2154')
# zones = gpd.read_file(ipath + r"MT_/Zones_Gironde_BX.gpkg").to_crs(epsg='2154')
zones['index'] = [f"zone_0{i}" if i < 10 else f"zone_{i}" for i in range(len(zones))]
zones['parking_fare'] = zones['parking_zone'].map(PARKING_FARES)

zones.set_index('index', inplace=True)
sm.zones = zones.copy()

centroids = zones.copy()
centroids['geometry'] = centroids['geometry'].apply(lambda x: x.centroid)
sm.centroids = centroids.copy()

# EXPORT

In [8]:
sm.to_zippedpickles(
    mpath + r'zones'
)

centroids: 100%|██████████| 13/13 [00:00<00:00, 15.86it/s] 


In [ ]:
end_of_notebook